# Controlled Algorithm Comparison
Compare K-Means, Ward hierarchical clustering, and DBSCAN using the same log-standard five-feature matrix. CustomerID is never a clustering feature.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'data/experiments/algorithm_comparison'
comparison = pd.read_csv(OUT / 'algorithm_comparison.csv')
dbscan = pd.read_csv(OUT / 'dbscan_results.csv')
profiles = pd.read_csv(OUT / 'selected_algorithm_profiles.csv')


## Shared preprocessing strategy
Frequency, MonetaryValue, and UniqueProducts use log1p; all five variables are standardized. Recency and CustomerLifetimeDays are not logged. Outliers are retained.

## K-Means baseline and hierarchical clustering
K-Means uses random_state=42 and n_init=20. Hierarchical clustering uses Ward linkage with Euclidean distance. DBSCAN metrics are based only on non-noise customers.

In [ ]:
fixed = comparison[comparison.algorithm != 'DBSCAN'].copy()
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, title in zip(axes, ['silhouette_score','davies_bouldin','calinski_harabasz'], ['Silhouette (higher)','Davies-Bouldin (lower)','Calinski-Harabasz (higher)']):
    labels = fixed.algorithm + ' ' + fixed.configuration.str.extract(r'(k=\d+)')[0]
    ax.bar(labels, fixed[metric], color=['#4472C4','#5B9BD5','#70AD47','#A5A5A5'])
    ax.set_title(title); ax.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()


## DBSCAN parameter exploration

In [ ]:
noise = dbscan.pivot(index='min_samples', columns='eps', values='noise_percentage')
fig, ax = plt.subplots(figsize=(9, 4))
image = ax.imshow(noise, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(noise.columns)), noise.columns); ax.set_yticks(range(len(noise.index)), noise.index)
ax.set_xlabel('eps'); ax.set_ylabel('min_samples'); ax.set_title('DBSCAN noise percentage')
fig.colorbar(image, ax=ax, label='% noise'); plt.show()


## Cluster-balance comparison

In [ ]:
shown = comparison[(comparison.algorithm != 'DBSCAN') | (comparison.configuration == 'eps=0.5, min_samples=20')].copy()
labels = shown.algorithm + ' ' + shown.configuration.str.extract(r'(k=\d+)')[0].fillna('selected')
plt.figure(figsize=(9, 4)); plt.bar(labels, shown.largest_cluster_percentage, label='Largest')
plt.bar(labels, shown.smallest_cluster_percentage, label='Smallest')
plt.ylabel('% of all customers'); plt.xticks(rotation=30); plt.legend(); plt.tight_layout(); plt.show()


## Behavioral interpretation and profile heatmaps
Profiles use medians in original customer units. DBSCAN noise is not treated as a customer segment.

In [ ]:
median_cols = [c for c in profiles if c.startswith('median_')]
matrix = profiles[median_cols].copy()
normalized = (matrix - matrix.mean()) / matrix.std(ddof=0)
fig, ax = plt.subplots(figsize=(12, 6)); image = ax.imshow(normalized, aspect='auto', cmap='coolwarm', vmin=-2, vmax=2)
ax.set_xticks(range(len(median_cols)), [c.replace('median_','') for c in median_cols], rotation=45, ha='right')
ax.set_yticks(range(len(profiles)), profiles.algorithm + ' | ' + profiles.configuration + ' | C' + profiles.cluster.astype(str))
ax.set_title('Original-unit median profiles (column z-scores for visualization)'); fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()


In [ ]:
km_h = profiles[profiles.algorithm.isin(['K-Means','Hierarchical'])]
for feature in ['median_Recency','median_Frequency','median_MonetaryValue']:
    plt.figure(figsize=(9,3)); plt.bar(km_h.algorithm + ' ' + km_h.configuration + ' C' + km_h.cluster.astype(str), km_h[feature])
    plt.title('K-Means vs Hierarchical: ' + feature.replace('median_','')); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


## Strongest quality-filtered DBSCAN profile

In [ ]:
db_profile = profiles[profiles.algorithm == 'DBSCAN']
db_profile[['configuration','cluster','customer_count','cluster_percentage',*median_cols]]


## Limitations and preliminary algorithm recommendation
Internal metrics do not measure business usefulness or future stability. DBSCAN metrics omit noise and are therefore not directly comparable with full-population scores. K-Means k=3 is the leading full-coverage candidate on metrics, balance, and behavioral clarity; k=4 remains a useful sensitivity case. This is a preliminary recommendation, not a final model decision.